# 邮件营销实验分析逐步复现

真实Hillstrom公开数据；以下文本输出由实际执行生成。完整重跑使用根目录`python run_all.py`。原数据不附带，首次读取会下载并校验。

In [1]:
from pathlib import Path
import os,sys
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd, numpy as np
from run_analysis import load_data,make_splits
from statistics_tools import mean_difference,ipw_score,policy_mask
from uplift_models import MetaLearner
df=load_data()
print(df.shape)
print(df.groupby('segment')[['visit','conversion','spend']].mean().round(6))

(64000, 13)
                  visit  conversion     spend
segment                                      
Mens E-Mail    0.182757    0.012531  1.422617
No E-Mail      0.106167    0.005726  0.652789
Womens E-Mail  0.151400    0.008837  1.077202


## SQL质量与组指标
每条记录作为一个原分配单元。没有客户ID，不随意去重。

In [2]:
import duckdb
con=duckdb.connect();con.register('customers',df)
print(con.execute((ROOT/'sql/01_arm_metrics.sql').read_text()).df().to_string(index=False))
print('Identical source rows retained:',df.drop(columns='record_id').duplicated().sum())

      segment     n  visitors  visit_rate  converters  conversion_rate  total_spend  revenue_per_assigned_customer  historical_spend_mean
  Mens E-Mail 21307    3894.0    0.182757       267.0         0.012531     30311.69                       1.422617             242.835931
    No E-Mail 21306    2262.0    0.106167       122.0         0.005726     13908.33                       0.652789             240.882653
Womens E-Mail 21387    3238.0    0.151400       189.0         0.008837     23038.11                       1.077202             242.536633
Identical source rows retained: 6562


## 两组均值差
二元结果均值是访问概率，绝对差乘100才是百分点。完整六项校正见导出表。

In [3]:
a=df.loc[df.segment=='Mens E-Mail','visit']
b=df.loc[df.segment=='No E-Mail','visit']
print(mean_difference(a,b))
print(pd.read_csv(ROOT/'reports/ab_effects.csv')[['treatment','outcome','effect','p_holm_six']].to_string(index=False))

{'effect': 0.07658956365153125, 'se': 0.0033858878117283653, 'ci_low': 0.06995314760976148, 'ci_high': 0.083225979693301, 'p_value': 1.3644511962742905e-112}
    treatment    outcome   effect    p_holm_six
  Mens E-Mail      visit 0.076590 8.186707e-112
  Mens E-Mail conversion 0.006805  6.007263e-13
  Mens E-Mail      spend 0.769827  3.491445e-07
Womens E-Mail      visit 0.045233  1.216224e-43
Womens E-Mail conversion 0.003111  3.120568e-04
Womens E-Mail      spend 0.424412  1.129397e-03


## CUPED思想的历史协变量回归调整
不是实验前同口径访问数据；方差收益小是实际结果。

In [4]:
print(pd.read_csv(ROOT/'reports/covariate_adjustment.csv')[['treatment','outcome','estimated_variance_reduction']].to_string(index=False))

    treatment    outcome  estimated_variance_reduction
  Mens E-Mail      visit                      0.005221
  Mens E-Mail conversion                      0.000672
  Mens E-Mail      spend                      0.000172
Womens E-Mail      visit                      0.004193
Womens E-Mail conversion                      0.000419
Womens E-Mail      spend                     -0.000006


## 一个可读的T-Learner例子
这里只重新拟合训练集模型并查看验证值，完整选模和测试流程在run_analysis.py中。

In [5]:
pair=df[df.segment.isin(['Mens E-Mail','No E-Mail'])].copy()
pair['treatment']=(pair.segment=='Mens E-Mail').astype(int)
train,val,test=make_splits(pair)
model=MetaLearner('t_logistic',20261107).fit(train,train.visit)
scores=model.predict(val)
mask=policy_mask(scores,.3)
z=ipw_score(val.visit,val.treatment)
print('Partitions:',len(train),len(val),len(test))
print('Validation incremental visits per 1000:',1000*np.mean(mask*z))

Partitions: 25567 8523 8523
Validation incremental visits per 1000: 23.700574914936055


## 最终测试结果与诚实解释
此处读取一键实验产生的最终报告，不用测试结果反向选模型。

In [6]:
print(pd.read_csv(ROOT/'reports/policy_evaluation.csv').round(4).to_string(index=False))
print(pd.read_csv(ROOT/'reports/policy_differences.csv').round(4).to_string(index=False))

           policy  fraction_contacted  incremental_visits_per_1000  ci_low  ci_high
     uplift_top30              0.2999                      36.6068 26.5165  46.6972
random30_expected              0.2999                      27.1640 22.2379  31.8790
   response_top30              0.2999                      38.2494 27.4551  48.8091
         send_all              1.0000                      90.5784 74.1523 106.3006
        send_none              0.0000                       0.0000  0.0000   0.0000
       comparator  difference_per_1000   ci_low  ci_high
random30_expected               9.4429   1.3937  17.2475
   response_top30              -1.6426 -12.0908   8.2131


![固定测试策略](../reports/figures/05_policy_values.png)

增量与普通响应排序的差值区间跨零，不能声称胜出。所有区间固定了训练模型与选中集合，没有覆盖全流程不确定性。

## 自己完成的扩展
1. 手算6条模拟记录的HT增量。
2. 在重新定义验证规则后研究20%与40%容量，不在同一个测试集反复择优。
3. 加入新的响应基线并写明结果。
4. 解释负增量预测不代表知道单个用户的真实反事实。